# Salary Sacrifice Cap Reform: Revenue and Distributional Impact Analysis

## Policy Context

Salary sacrifice pension schemes allow employees to exchange part of their salary for employer pension contributions, reducing both income tax and National Insurance contributions. This reform proposes capping salary sacrifice contributions at £2,000 per year.

### How the Reform Works

For individuals with salary sacrifice pension contributions exceeding £2,000:
1. **Excess contributions** above £2,000 are redirected to employment income
2. **Employer response**: 13% haircut applied (employers retain 13% of the excess, employees receive 87%)
3. **Employee pension contributions** increase to maintain some retirement savings

This notebook implements and analyzes this structural reform using the PolicyEngine UK microsimulation model.

## Reform Implementation

### Key Assumptions

1. **Cap Amount**: £2,000 per year on salary sacrifice pension contributions
2. **Employer Response Haircut**: 13% of excess contributions retained by employers
3. **Employee Adjustment**: 87% of excess becomes employment income
4. **Pension Preservation**: Redirected amounts also added to employee pension contributions

### Implementation Logic

For each year 2026-2030:
```python
excess_ss = max(salary_sacrifice - £2,000, 0)
new_employment_income = employment_income + excess_ss × 0.87
new_employee_pension = employee_pension + excess_ss × 0.87
new_salary_sacrifice = min(salary_sacrifice, £2,000)
```

In [1]:
# Import required libraries
from policyengine_uk import Microsimulation, Scenario
from policyengine_uk.model_api import *
from policyengine_uk.data import UKSingleYearDataset
import numpy as np

/Users/janansadeqian/anaconda3/envs/python313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_ss_cap_reform(employer_response_haircut: float = 0.13, cap_amount: float = 2000):
    """
    Structural reform: Cap salary sacrifice pension contributions.
    
    Args:
        employer_response_haircut: Fraction of excess contributions retained by employers (default: 0.13)
        cap_amount: Maximum allowed salary sacrifice contributions in GBP (default: 2000)
    
    This reform caps salary sacrifice contributions and redirects excess to:
    1. Employment income (subject to income tax and NICs)
    2. Employee pension contributions (maintaining some retirement savings)
    
    Employers retain a portion (haircut) of the excess, reflecting labor market dynamics.
    """
    
    def modify(sim):
        for year in range(2026, 2031):
            # Calculate excess salary sacrifice contributions
            ss_contrib = sim.calculate("pension_contributions_via_salary_sacrifice", period=year).values
            excess_ss_contrib = np.maximum(ss_contrib - cap_amount, 0)
            
            # Redirect excess to employment income (with employer haircut)
            emp_income = sim.calculate("employment_income", period=year).values
            new_employment_income = emp_income + excess_ss_contrib * (1 - employer_response_haircut)
            
            # Update employee pension contributions
            employee_pension = sim.calculate("employee_pension_contributions", period=year).values
            new_employee_pension = employee_pension + excess_ss_contrib * (1 - employer_response_haircut)
            
            # Apply the reformed values
            sim.set_input("employment_income", year, new_employment_income)
            sim.set_input("employee_pension_contributions", year, new_employee_pension)
            sim.set_input("pension_contributions_via_salary_sacrifice", year, ss_contrib - excess_ss_contrib)
    
    return Scenario(simulation_modifier=modify)

# Create the reform scenario
ss_cap_reform = create_ss_cap_reform(employer_response_haircut=0.13, cap_amount=2000)

## Data Loading and Simulation

In [3]:
# Load local dataset
dataset = UKSingleYearDataset(
    file_path="/Users/janansadeqian/policyengine-uk-data/policyengine_uk_data/storage/enhanced_frs_2023_24.h5"
)

# Create baseline and reformed microsimulations
print("Loading microsimulation data...")
baseline = Microsimulation(dataset=dataset)
reformed = Microsimulation(dataset=dataset, scenario=ss_cap_reform)
print("Done!")

Loading microsimulation data...
Done!


## Revenue Impact Analysis

This section calculates the budgetary impact of the salary sacrifice cap, including:
- Changes in income tax revenue
- Changes in National Insurance contributions
- Total revenue gain
- Number of affected individuals

In [4]:
# Calculate revenue impacts
year = 2026

# Income tax revenue
baseline_income_tax = baseline.calculate("income_tax", year).sum() / 1e9
reformed_income_tax = reformed.calculate("income_tax", year).sum() / 1e9
income_tax_gain = reformed_income_tax - baseline_income_tax

# National Insurance contributions
baseline_ni = baseline.calculate("national_insurance", year).sum() / 1e9
reformed_ni = reformed.calculate("national_insurance", year).sum() / 1e9
ni_gain = reformed_ni - baseline_ni

# Total revenue gain
total_revenue_gain = income_tax_gain + ni_gain

# Affected individuals
baseline_ss = baseline.calculate("pension_contributions_via_salary_sacrifice", year)
affected_individuals = (baseline_ss > 2000).sum()
total_individuals = len(baseline_ss)

# Average impact per affected individual
if affected_individuals > 0:
    baseline_net_income = baseline.calculate("household_net_income", year)
    reformed_net_income = reformed.calculate("household_net_income", year)
    
    # Get household members with excess salary sacrifice
    person_to_household = baseline.calculate("household_id", year)
    affected_households = set(person_to_household[baseline_ss > 2000])
    
    avg_loss_per_affected = total_revenue_gain * 1e9 / affected_individuals
else:
    avg_loss_per_affected = 0

print("=" * 70)
print("BUDGETARY IMPACT: SALARY SACRIFICE CAP (£2,000)")
print("=" * 70)
print(f"\nYear: {year}")
print(f"\nRevenue Effects:")
print(f"  Income Tax Gain:         £{income_tax_gain:.2f}bn")
print(f"  National Insurance Gain: £{ni_gain:.2f}bn")
print(f"  " + "-" * 50)
print(f"  {'TOTAL REVENUE GAIN:':<25} £{total_revenue_gain:.2f} billion")
print(f"\nAffected Population:")
print(f"  Individuals with SS > £2,000: {affected_individuals:,.0f} ({affected_individuals/total_individuals*100:.2f}%)")
print(f"  Average loss per affected:    £{avg_loss_per_affected:.0f} per year")
print(f"  Average monthly loss:         £{avg_loss_per_affected/12:.2f}")
print("\n" + "=" * 70)

BUDGETARY IMPACT: SALARY SACRIFICE CAP (£2,000)

Year: 2026

Revenue Effects:
  Income Tax Gain:         £0.89bn
  National Insurance Gain: £0.06bn
  --------------------------------------------------
  TOTAL REVENUE GAIN:       £0.95 billion

Affected Population:
  Individuals with SS > £2,000: 409,777 (354.44%)
  Average loss per affected:    £2308 per year
  Average monthly loss:         £192.32



## Distributional Analysis

This section analyzes how the reform affects different income groups.

In [5]:
# Calculate distributional impacts
baseline_net = baseline.calculate("household_net_income", year)
reformed_net = reformed.calculate("household_net_income", year)
income_change = reformed_net - baseline_net

# Get household weights
weights = baseline.calculate("household_weight", year)

# Calculate by income decile
import pandas as pd

# Create deciles based on baseline income
decile_labels = []
decile_avg_loss = []
decile_pct_affected = []

for decile in range(1, 11):
    # Get households in this decile
    decile_mask = (pd.qcut(baseline_net, 10, labels=False, duplicates='drop') == (decile - 1))
    
    if decile_mask.sum() > 0:
        decile_income_change = income_change[decile_mask]
        decile_weights = weights[decile_mask]
        
        # Calculate weighted average loss
        weighted_loss = (decile_income_change * decile_weights).sum() / decile_weights.sum()
        
        # Calculate percentage affected (those with any loss)
        affected = (decile_income_change < -1).sum()  # More than £1 loss
        pct_affected = affected / decile_mask.sum() * 100
        
        decile_labels.append(f"Decile {decile}")
        decile_avg_loss.append(weighted_loss)
        decile_pct_affected.append(pct_affected)

print("=" * 80)
print("DISTRIBUTIONAL IMPACT BY INCOME DECILE")
print("=" * 80)
print(f"\n{'Decile':<15} {'Avg Loss (£/year)':<25} {'% Affected':<20}")
print("-" * 80)

for i, label in enumerate(decile_labels):
    loss_str = f"£{abs(decile_avg_loss[i]):.2f}" if decile_avg_loss[i] < 0 else f"+£{decile_avg_loss[i]:.2f}"
    print(f"{label:<15} {loss_str:<25} {decile_pct_affected[i]:.1f}%")

print("\n" + "=" * 80)

DISTRIBUTIONAL IMPACT BY INCOME DECILE

Decile          Avg Loss (£/year)         % Affected          
--------------------------------------------------------------------------------
Decile 1        +£0.82                    0.0%
Decile 2        +£0.01                    0.0%
Decile 3        +£0.07                    0.0%
Decile 4        +£0.43                    0.0%
Decile 5        +£6.71                    0.0%
Decile 6        +£2.27                    0.0%
Decile 7        +£33.06                   0.0%
Decile 8        +£39.12                   0.0%
Decile 9        +£270.78                  0.0%
Decile 10       +£1295.21                 0.0%



## Sensitivity Analysis

Test different cap amounts and employer response rates.

In [6]:
# Test different cap amounts
cap_amounts = [1000, 2000, 3000, 5000]
results = []

print("Testing different cap amounts (employer haircut = 13%)...\n")

for cap in cap_amounts:
    reform = create_ss_cap_reform(employer_response_haircut=0.13, cap_amount=cap)
    reformed_sim = Microsimulation(dataset=dataset, scenario=reform)
    
    # Calculate revenue gain
    reformed_it = reformed_sim.calculate("income_tax", year).sum() / 1e9
    reformed_ni = reformed_sim.calculate("national_insurance", year).sum() / 1e9
    revenue_gain = (reformed_it - baseline_income_tax) + (reformed_ni - baseline_ni)
    
    # Calculate affected individuals
    baseline_ss = baseline.calculate("pension_contributions_via_salary_sacrifice", year)
    affected = (baseline_ss > cap).sum()
    
    results.append({
        'cap': cap,
        'revenue': revenue_gain,
        'affected': affected
    })

print("=" * 70)
print("SENSITIVITY ANALYSIS: CAP AMOUNT")
print("=" * 70)
print(f"\n{'Cap Amount':<15} {'Revenue Gain (£bn)':<25} {'Affected Individuals':<20}")
print("-" * 70)

for r in results:
    print(f"£{r['cap']:<14} £{r['revenue']:.2f}bn{'':<18} {r['affected']:,.0f}")

print("\n" + "=" * 70)

Testing different cap amounts (employer haircut = 13%)...

SENSITIVITY ANALYSIS: CAP AMOUNT

Cap Amount      Revenue Gain (£bn)        Affected Individuals
----------------------------------------------------------------------
£1000           £1.01bn                   578,576
£2000           £0.95bn                   409,777
£3000           £0.89bn                   294,247
£5000           £0.81bn                   178,971



## Key Findings

### Policy Implications

1. **Revenue Generation**: The reform generates significant tax revenue by limiting tax-advantaged pension contributions
2. **Targeting**: Primarily affects higher earners who can afford larger salary sacrifice contributions
3. **Trade-offs**: Balances revenue generation with maintaining retirement savings incentives

### Considerations

- **Employer Response**: 13% haircut assumes employers retain some savings; actual behavior may vary
- **Behavioral Effects**: High earners may seek alternative tax-efficient savings vehicles
- **Retirement Adequacy**: May reduce pension savings for affected individuals
- **Administrative Complexity**: Requires tracking and enforcement of contribution limits